# Mercari Price Prediction — SHAP Açıklanabilirlik Analizi

Modelin tahminlerini açıklıyoruz:
- Hangi feature'lar fiyatı en çok etkiliyor?
- Tek bir ürün için neden bu fiyatı tahmin etti?
- Kategoriler arası fark nereden geliyor?


In [ ]:
import os
os.chdir('/home/cevdetkopuz/mercari-price-prediction')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
import joblib
import warnings
warnings.filterwarnings('ignore')

# Pipeline yükle
tfidf_name = joblib.load('models/tfidf_name.joblib')
tfidf_desc = joblib.load('models/tfidf_desc.joblib')
le_cat_main = joblib.load('models/le_cat_main.joblib')
le_cat_sub1 = joblib.load('models/le_cat_sub1.joblib')
le_cat_sub2 = joblib.load('models/le_cat_sub2.joblib')
le_brand = joblib.load('models/le_brand.joblib')
top_brands = joblib.load('models/top_brands.joblib')
ridge_model = joblib.load('models/ridge_model.joblib')

print("Pipeline yüklendi")


## 1. Veriyi Hazırla (Küçük Örneklem)

SHAP hesaplaması yoğun. 1.48M satırla çalışmak yerine 
**10K örneklem** alacağız — sonuçlar istatistiksel olarak yeterli.

In [ ]:
# Veri yükle ve preprocessing
df = pd.read_csv('data/raw/train.tsv', sep='\t')
df = df[df['price'] > 0].reset_index(drop=True)

df['brand_name'].fillna('missing', inplace=True)
df['category_name'].fillna('missing/missing/missing', inplace=True)
df['item_description'].fillna('No description yet', inplace=True)

def split_category(cat):
    parts = str(cat).split('/')
    while len(parts) < 3:
        parts.append('missing')
    return parts[:3]

cat_split = df['category_name'].apply(split_category)
df['cat_main'] = cat_split.apply(lambda x: x[0])
df['cat_sub1'] = cat_split.apply(lambda x: x[1])
df['cat_sub2'] = cat_split.apply(lambda x: x[2])

df['brand_name'] = df['brand_name'].apply(lambda x: x if x in top_brands else 'other')
df['cat_main_enc'] = le_cat_main.transform(df['cat_main'])
df['cat_sub1_enc'] = le_cat_sub1.transform(df['cat_sub1'])
df['cat_sub2_enc'] = le_cat_sub2.transform(df['cat_sub2'])
df['brand_enc'] = le_brand.transform(df['brand_name'])
df['desc_word_count'] = df['item_description'].str.split().str.len()
df['name_len'] = df['name'].str.len()
df['has_description'] = (df['item_description'] != 'No description yet').astype(int)

numeric_features = ['item_condition_id', 'shipping', 'cat_main_enc',
                    'cat_sub1_enc', 'cat_sub2_enc', 'brand_enc',
                    'desc_word_count', 'name_len', 'has_description']

y = np.log1p(df['price'])

print(f"Toplam veri: {len(df):,}")


In [ ]:
# 10K örneklem al (tekrarlanabilir)
np.random.seed(42)
sample_idx = np.random.choice(len(df), 10000, replace=False)
df_sample = df.iloc[sample_idx].reset_index(drop=True)
y_sample = y.iloc[sample_idx].reset_index(drop=True)

# Feature matrisi (sadece örneklem için)
X_numeric = csr_matrix(df_sample[numeric_features].values, dtype=np.float32)
X_name = tfidf_name.transform(df_sample['name'])
X_desc = tfidf_desc.transform(df_sample['item_description'])
X_sample = hstack([X_numeric, X_name, X_desc]).tocsr()

print(f"Örneklem: {X_sample.shape[0]:,} satır × {X_sample.shape[1]:,} feature")


## 2. SHAP: Sayısal Feature'lar (Hızlı Analiz)

Önce sadece 9 sayısal feature'a bakalım — bu çok hızlı ve anlaşılır.
TF-IDF feature'ları 100K tane olduğu için onları ayrı ele alacağız.

In [ ]:
# Sadece sayısal feature'larla küçük bir Ridge eğit
X_num_only = df_sample[numeric_features].values.astype(np.float32)

ridge_numeric = Ridge(alpha=5.0)
ridge_numeric.fit(X_num_only, y_sample)

print(f"Sayısal model R²: {ridge_numeric.score(X_num_only, y_sample):.4f}")
print("(Bu düşük olabilir — TF-IDF olmadan model zayıf, ama feature etkilerini gösteriyor)")


In [ ]:
# SHAP hesapla (LinearExplainer — Ridge için optimize)
print("SHAP değerleri hesaplanıyor...")
explainer = shap.LinearExplainer(ridge_numeric, X_num_only)
shap_values = explainer.shap_values(X_num_only)

print(f"SHAP matrisi: {shap_values.shape}")
print("Tamamlandı!")


## 3. SHAP Görselleştirmeleri

### 3.1 Feature Importance — Hangi feature en etkili?

In [ ]:
# Summary plot — bar (genel önem sıralaması)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_num_only, 
                  feature_names=numeric_features,
                  plot_type="bar", show=False)
plt.title('Feature Importance (ortalama SHAP etkisi)')
plt.tight_layout()
plt.savefig('data/processed/13_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.2 Beeswarm Plot — Feature değerleri fiyatı nasıl etkiliyor?

Her nokta bir ürün. Renk = feature değeri (kırmızı = yüksek, mavi = düşük).
Sağa giden noktalar fiyatı artırıyor, sola gidenler düşürüyor.

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_num_only,
                  feature_names=numeric_features, show=False)
plt.title('SHAP Beeswarm — Feature değeri vs fiyat etkisi')
plt.tight_layout()
plt.savefig('data/processed/14_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.3 Tek Ürün Açıklaması — Waterfall Plot

Rastgele bir ürün seçip, modelin neden bu fiyatı tahmin ettiğini görelim.

In [ ]:
# İlginç bir ürün seç (ortalamanın üstünde fiyatlı)
interesting_idx = df_sample[df_sample['price'] > 50].sample(1, random_state=42).index[0]

print(f"Seçilen ürün:")
print(f"  Ad: {df_sample.loc[interesting_idx, 'name']}")
print(f"  Kategori: {df_sample.loc[interesting_idx, 'cat_main']}")
print(f"  Marka: {df_sample.loc[interesting_idx, 'brand_name']}")
print(f"  Durum: {df_sample.loc[interesting_idx, 'item_condition_id']}")
print(f"  Kargo: {'Satıcı öder' if df_sample.loc[interesting_idx, 'shipping'] == 1 else 'Alıcı öder'}")
print(f"  Gerçek fiyat: ${df_sample.loc[interesting_idx, 'price']:.2f}")

pred = ridge_numeric.predict(X_num_only[interesting_idx:interesting_idx+1])[0]
print(f"  Tahmin (sadece sayısal): ${np.expm1(pred):.2f}")
print()

# Waterfall plot
shap_explanation = shap.Explanation(
    values=shap_values[interesting_idx],
    base_values=explainer.expected_value,
    feature_names=numeric_features,
    data=X_num_only[interesting_idx]
)

plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_explanation, show=False)
plt.title('Waterfall: Bu ürünün fiyatını ne etkiledi?')
plt.tight_layout()
plt.savefig('data/processed/15_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.4 Dependence Plot — Shipping ve Brand etkisi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Shipping etkisi
shap.dependence_plot('shipping', shap_values, X_num_only,
                     feature_names=numeric_features,
                     ax=axes[0], show=False)
axes[0].set_title('Shipping: SHAP etkisi')

# Brand etkisi
shap.dependence_plot('brand_enc', shap_values, X_num_only,
                     feature_names=numeric_features,
                     ax=axes[1], show=False)
axes[1].set_title('Brand encoding: SHAP etkisi')

plt.tight_layout()
plt.savefig('data/processed/16_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. TF-IDF Feature Analizi

100K TF-IDF feature'ın hepsine SHAP uygulamak çok ağır.
Bunun yerine Ridge model katsayılarına bakacağız — doğrusal modelde
katsayı = o kelimenin fiyata etkisi.

In [ ]:
# Ridge katsayıları — hangi kelimeler fiyatı artırıyor/düşürüyor?

# Feature isimleri
name_features = tfidf_name.get_feature_names_out()
desc_features = tfidf_desc.get_feature_names_out()

# Ridge modelinin katsayıları (tam modelden)
coefs = ridge_model.coef_

# İlk 9 katsayı sayısal feature'lar, sonraki 50K name, sonraki 50K desc
numeric_coefs = coefs[:9]
name_coefs = coefs[9:50009]
desc_coefs = coefs[50009:]

print("=== Sayısal Feature Katsayıları ===")
for feat, coef in sorted(zip(numeric_features, numeric_coefs), key=lambda x: abs(x[1]), reverse=True):
    direction = "artırır" if coef > 0 else "düşürür"
    print(f"  {feat:25s}: {coef:+.4f} (fiyatı {direction})")


In [ ]:
# Name'deki en etkili kelimeler
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Fiyatı EN ÇOK ARTIRAN name kelimeleri
top_positive_idx = np.argsort(name_coefs)[-20:]
top_positive_words = name_features[top_positive_idx]
top_positive_vals = name_coefs[top_positive_idx]

axes[0].barh(range(20), top_positive_vals, color='#1D9E75', alpha=0.8)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(top_positive_words)
axes[0].set_title('Ürün adında fiyatı ARTIRAN kelimeler')
axes[0].set_xlabel('Ridge katsayısı')

# Fiyatı EN ÇOK DÜŞÜREN name kelimeleri
top_negative_idx = np.argsort(name_coefs)[:20]
top_negative_words = name_features[top_negative_idx]
top_negative_vals = name_coefs[top_negative_idx]

axes[1].barh(range(20), top_negative_vals, color='#E24B4A', alpha=0.8)
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(top_negative_words)
axes[1].set_title('Ürün adında fiyatı DÜŞÜREN kelimeler')
axes[1].set_xlabel('Ridge katsayısı')

plt.tight_layout()
plt.savefig('data/processed/17_name_word_impact.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Description'daki en etkili kelimeler
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Fiyatı artıran description kelimeleri
top_pos_idx = np.argsort(desc_coefs)[-20:]
axes[0].barh(range(20), desc_coefs[top_pos_idx], color='#1D9E75', alpha=0.8)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(desc_features[top_pos_idx])
axes[0].set_title('Açıklamada fiyatı ARTIRAN kelimeler')
axes[0].set_xlabel('Ridge katsayısı')

# Fiyatı düşüren description kelimeleri
top_neg_idx = np.argsort(desc_coefs)[:20]
axes[1].barh(range(20), desc_coefs[top_neg_idx], color='#E24B4A', alpha=0.8)
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(desc_features[top_neg_idx])
axes[1].set_title('Açıklamada fiyatı DÜŞÜREN kelimeler')
axes[1].set_xlabel('Ridge katsayısı')

plt.tight_layout()
plt.savefig('data/processed/18_desc_word_impact.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Sonuç ve Çıkarımlar

### SHAP analizinden öğrendiklerimiz:
1. **En etkili feature'lar** → summary plot'tan görüyoruz
2. **Shipping etkisi** → satıcı öderse fiyat düşüyor (beklenen)
3. **Brand etkisi** → bazı brand encoding'leri fiyatı belirgin artırıyor
4. **Kelime bazlı etki** → "authentic", "rare" gibi kelimeler fiyatı artırır, "free", "old" düşürür

### Bu bilgilerle ne yapabiliriz (Ideal versiyonda):
- Önemsiz feature'ları çıkarabiliriz (feature selection)
- Fiyatı artıran/düşüren kelimeleri UI'da kullanıcıya gösterebiliriz
- Dream versiyonda LLM'e "bu kelimeler etkili" bilgisini verebiliriz
